In [1]:
import numpy as np
import pandas as pd
import os
import glob
import warnings
warnings.filterwarnings('ignore') 

BASE_DIR = '/kaggle/input/datasets/jeanmidev/smart-meters-in-london/'

print("Loading demographic and weather metadata...")
info_df = pd.read_csv(os.path.join(BASE_DIR, 'informations_households.csv'))
weather_df = pd.read_csv(os.path.join(BASE_DIR, 'weather_hourly_darksky.csv'))

weather_df['time'] = pd.to_datetime(weather_df['time'])

block_0_path = os.path.join(BASE_DIR, 'halfhourly_dataset/halfhourly_dataset/block_0.csv')
print(f"Target block file found at: {block_0_path}")

def process_energy_block(block_path, info_data, weather_data):
    """
    Loads a block file, cleans it, and merges demographic/weather context.
    """
    print(f"Reading energy block data from {block_path}...")
    block = pd.read_csv(block_path)
    
    print("Formatting timestamps and cleaning missing values...")
    block['energy(kWh/hh)'] = pd.to_numeric(block['energy(kWh/hh)'], errors='coerce')
    block['tstp'] = pd.to_datetime(block['tstp'])
    
    print("Merging ACORN demographic data...")
    block = pd.merge(block, info_data[['LCLid', 'Acorn', 'Acorn_grouped']], on='LCLid', how='left')
    
    block['weather_time_match'] = block['tstp'].dt.floor('h')

    print("Merging hourly darksky weather data...")
    merged = pd.merge(block, weather_data, left_on='weather_time_match', right_on='time', how='left')
    
    merged.drop(columns=['weather_time_match', 'time'], inplace=True)
    
    merged = merged.sort_values(by=['LCLid', 'tstp'])
    merged['energy(kWh/hh)'] = merged.groupby('LCLid')['energy(kWh/hh)'].transform(lambda x: x.ffill().bfill())
    
    return merged

print("\n--- Starting Data Pipeline ---")
df = process_energy_block(block_0_path, info_df, weather_df)

print("\n--- Phase 1 Complete ---")
print("Sanitized Dataframe Shape:", df.shape)
print("\nFirst 5 rows:")
print(df[['LCLid', 'tstp', 'energy(kWh/hh)', 'Acorn_grouped', 'temperature']].head())

Loading demographic and weather metadata...
Target block file found at: /kaggle/input/datasets/jeanmidev/smart-meters-in-london/halfhourly_dataset/halfhourly_dataset/block_0.csv

--- Starting Data Pipeline ---
Reading energy block data from /kaggle/input/datasets/jeanmidev/smart-meters-in-london/halfhourly_dataset/halfhourly_dataset/block_0.csv...
Formatting timestamps and cleaning missing values...
Merging ACORN demographic data...
Merging hourly darksky weather data...

--- Phase 1 Complete ---
Sanitized Dataframe Shape: (1222670, 16)

First 5 rows:
       LCLid                tstp  energy(kWh/hh) Acorn_grouped  temperature
0  MAC000002 2012-10-12 00:30:00             0.0      Affluent        13.61
1  MAC000002 2012-10-12 01:00:00             0.0      Affluent        13.09
2  MAC000002 2012-10-12 01:30:00             0.0      Affluent        13.09
3  MAC000002 2012-10-12 02:00:00             0.0      Affluent        12.54
4  MAC000002 2012-10-12 02:30:00             0.0      Affluent

In [6]:
# --- Phase 2: Feature Engineering ---

print("Extracting temporal features...")

df['hour'] = df['tstp'].dt.hour
df['day_of_week'] = df['tstp'].dt.dayofweek
df['month'] = df['tstp'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print("Calculating historical lags...")

df = df.sort_values(by=['LCLid', 'tstp']).reset_index(drop=True)

df['energy_lag_1h'] = df.groupby('LCLid')['energy(kWh/hh)'].shift(2)

df['energy_lag_24h'] = df.groupby('LCLid')['energy(kWh/hh)'].shift(48)

df['energy_rolling_mean_24h'] = df.groupby('LCLid')['energy(kWh/hh)'].transform(lambda x: x.rolling(48).mean())

df.dropna(inplace=True)

print("\n--- Phase 2 Complete ---")
print("New Dataframe Shape:", df.shape)
print("\nFirst 5 rows of engineered features:")
print(df[['LCLid', 'tstp', 'energy(kWh/hh)', 'energy_lag_1h', 'energy_lag_24h', 'hour']].head())

Extracting temporal features...
Calculating historical lags...

--- Phase 2 Complete ---
New Dataframe Shape: (1216724, 23)

First 5 rows of engineered features:
        LCLid                tstp  energy(kWh/hh)  energy_lag_1h  \
48  MAC000002 2012-10-14 01:00:00           0.226          0.262   
49  MAC000002 2012-10-14 01:30:00           0.088          0.166   
50  MAC000002 2012-10-14 02:00:00           0.126          0.226   
51  MAC000002 2012-10-14 02:30:00           0.082          0.088   
52  MAC000002 2012-10-14 03:00:00           0.123          0.126   

    energy_lag_24h  hour  
48           0.275     1  
49           0.256     1  
50           0.211     2  
51           0.136     2  
52           0.161     3  


In [5]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

print("--- Starting Phase 3: AI Modeling ---")

features = ['hour', 'day_of_week', 'month', 'is_weekend', 
            'temperature', 'energy_lag_1h', 'energy_lag_24h', 'energy_rolling_mean_24h']
target = 'energy(kWh/hh)'

df = df.dropna(subset=features + [target])

X = df[features]
y = df[target]

split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training on {len(X_train)} historical records...")
print(f"Testing on {len(X_test)} future records...")

model = xgb.XGBRegressor(
    n_estimators=150, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42,
    n_jobs=-1 
)

print("Training model...")
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

print("\n--- Model Evaluation ---")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f} kWh")
print(f"MAE (Mean Absolute Error): {mae:.4f} kWh")

def autonomous_grid_decision(predicted_load, safety_threshold=1.5):
    """
    Simulates the platform's automated response to predicted grid loads.
    """
    if predicted_load > safety_threshold:
        return "ALERT: High Load Forecasted. Action: Initiating Peak Shaving & Discharging Battery Storage."
    elif predicted_load < 0.2:
        return "STATUS: Low Load. Action: Routing excess renewable generation to storage."
    else:
        return "STATUS: Grid Stable. Action: Maintain standard distribution."

sample_prediction = predictions[0]
decision = autonomous_grid_decision(sample_prediction)

print("\n--- Autonomous Platform Output ---")
print(f"Forecasted Household Load: {sample_prediction:.4f} kWh")
print(f"Platform Decision: {decision}")

--- Starting Phase 3: AI Modeling ---
Training on 975299 historical records...
Testing on 243825 future records...
Training model...

--- Model Evaluation ---
RMSE (Root Mean Squared Error): 0.3319 kWh
MAE (Mean Absolute Error): 0.1669 kWh

--- Autonomous Platform Output ---
Forecasted Household Load: 0.1572 kWh
Platform Decision: STATUS: Low Load. Action: Routing excess renewable generation to storage.


In [4]:
import joblib
import shap
import json

model.save_model('xgboost_energy_model.json')
joblib.dump(features, 'model_features.pkl')

explainer = shap.Explainer(model, X_train.sample(1000)) # Sample for speed
shap_values = explainer(X_test.sample(1000))

joblib.dump(shap_values, 'shap_values.pkl')

metadata = {
    "model_type": "XGBRegressor",
    "rmse": float(rmse),
    "mae": float(mae),
    "features": features
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f)

print("--- System Export Complete ---")
print("1. Download 'xgboost_energy_model.json'")
print("2. Download 'model_features.pkl'")
print("3. Download 'shap_values.pkl'")
print("4. Download 'model_metadata.json'")

--- System Export Complete ---
1. Download 'xgboost_energy_model.json'
2. Download 'model_features.pkl'
3. Download 'shap_values.pkl'
4. Download 'model_metadata.json'
